In [ ]:
!wget "https://next.phytlsigns.com/index.php/s/R89wDWHrW2Zzomb/download"

--2025-12-20 11:36:04--  https://next.phytlsigns.com/index.php/s/R89wDWHrW2Zzomb/download
Resolving next.phytlsigns.com (next.phytlsigns.com)... 81.17.18.218
Connecting to next.phytlsigns.com (next.phytlsigns.com)|81.17.18.218|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [application/zip]
Saving to: ‘download’

download                [     <=>            ]  23.73G  25.0MB/s    in 17m 21s 

2025-12-20 11:53:26 (23.3 MB/s) - ‘download’ saved [25479327673]



In [ ]:
!unzip "download"

Archive:  download
   creating: General Stress Data and Paper/
 extracting: General Stress Data and Paper/General Stress Paper.pdf  
 extracting: General Stress Data and Paper/GeneralStressPaperData.tar  
 extracting: General Stress Data and Paper/GeneralStress_SuppMat.pdf  
 extracting: General Stress Data and Paper/Readme (copy).txt  


In [ ]:
!tar -xf "/content/General Stress Data and Paper/GeneralStressPaperData.tar"

In [ ]:
!rm -f "/content/General Stress Data and Paper/GeneralStressPaperData.tar"


# Preprocessing pipeline

In [ ]:
import os, glob
import numpy as np
import h5py
from scipy.signal import iirnotch, filtfilt
import scipy.stats as stats
import pywt


# parameters
BASE_DIR = "/content/GeneralStressPaperData"
FS = 500.0                                   # sampling rate (Hz)

L_SECONDS = 60        # window length l in seconds
N_WINDOWS = 15        # number of windows N in each segment

# for comparing different L_SECONDS values
USE_274_PER_PLANT = False
SEED = 123

print("BASE_DIR:", BASE_DIR)


BASE_DIR: /content/GeneralStressPaperData


## 1) Notch filtering (50 Hz and 100 Hz)

In [ ]:
# Notch filter
Q = 30.0  # bandwidth control
b50, a50 = iirnotch(50.0 / (FS / 2), Q)
b100, a100 = iirnotch(100.0 / (FS / 2), Q)

def apply_notch_filters(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = filtfilt(b50, a50, x)
    x = filtfilt(b100, a100, x)
    return x


In [ ]:
mat_files = sorted(glob.glob(os.path.join(BASE_DIR, "*.mat")))
print("Found", len(mat_files), "MAT files")

def load_mv_signal_mat(path: str) -> np.ndarray:
    with h5py.File(path, "r") as f:
        return np.array(f["mV"]).squeeze()

for path in mat_files:
    out_path = path.replace(".mat", "_filtered.npy")
    if os.path.exists(out_path):
        continue
    raw = load_mv_signal_mat(path)
    filt = apply_notch_filters(raw)
    np.save(out_path, filt)
    del raw, filt

print("Done. Filtered files now exist as *_filtered.npy")


Found 36 MAT files
Done. Filtered files now exist as *_filtered.npy


## 2) Feature extraction (34 features per window)

In [ ]:
def _hjorth_params(x: np.ndarray):
    x = np.asarray(x, dtype=np.float64)
    if x.size < 3:
        return 0.0, 0.0
    dx = np.diff(x)
    ddx = np.diff(dx)
    var_x = np.var(x)
    var_dx = np.var(dx)
    var_ddx = np.var(ddx)
    mobility = np.sqrt(var_dx / var_x) if var_x > 0 else 0.0
    complexity = (np.sqrt(var_ddx / var_dx) / mobility) if (var_dx > 0 and mobility > 0) else 0.0
    return mobility, complexity

def _ghe_q1(x: np.ndarray, min_tau=2, max_tau=256):
    """Generalized Hurst Exponent for q=1 (fast approximation)."""
    x = np.asarray(x, dtype=np.float64)
    n = x.size
    max_tau = int(min(max_tau, n // 4))
    min_tau = int(max(1, min_tau))
    if max_tau <= min_tau + 1:
        return 0.5
    taus = []
    vals = []
    tau = min_tau
    while tau <= max_tau:
        d = np.abs(x[tau:] - x[:-tau])
        v = np.mean(d)
        if np.isfinite(v) and v > 0:
            taus.append(tau)
            vals.append(v)
        tau *= 2
    if len(taus) < 2:
        return 0.5
    log_t = np.log(np.asarray(taus))
    log_v = np.log(np.asarray(vals))
    slope = np.polyfit(log_t, log_v, 1)[0]
    H = slope  # for q=1, E(|x(t+tau)-x(t)|) ~ tau^H
    return float(np.clip(H, 0.0, 1.5))

def _shannon_entropy(p: np.ndarray):
    p = np.asarray(p, dtype=np.float64)
    p = p[p > 0]
    if p.size == 0:
        return 0.0
    p = p / np.sum(p)
    return float(-np.sum(p * np.log2(p)))

def _wavelet_packet_entropy(x: np.ndarray, wavelet='db8', level=3):
    """Shannon entropy of wavelet packet energies (proxy for wentropy)."""
    wp = pywt.WaveletPacket(data=x, wavelet=wavelet, mode='symmetric', maxlevel=level)
    nodes = wp.get_level(level, order='freq')
    energies = np.array([np.sum(np.square(n.data)) for n in nodes], dtype=np.float64)
    if np.sum(energies) == 0:
        return 0.0
    return _shannon_entropy(energies)

def _spectral_features(x: np.ndarray, fs: float):
    # PSD via FFT magnitude (fast)
    x = np.asarray(x, dtype=np.float64)
    n = x.size
    if n < 4:
        return 0.0, 0.0, 0.0
    X = np.fft.rfft(x - np.mean(x))
    P = np.abs(X) ** 2
    freqs = np.fft.rfftfreq(n, d=1/fs)
    Psum = np.sum(P)
    if Psum <= 0:
        return 0.0, 0.0, 0.0
    centroid = float(np.sum(freqs * P) / Psum)  # frequency center
    frms = float(np.sqrt(np.sum((freqs**2) * P) / Psum))  # frequency RMS
    fvar = float(np.sqrt(np.sum(((freqs - centroid)**2) * P) / Psum))  # root variance (std)
    return centroid, frms, fvar

def _color_noise_similarity(x: np.ndarray, fs: float):
    """Similarity between signal PSD and colored noise PSD slopes."""
    x = np.asarray(x, dtype=np.float64)
    n = x.size
    if n < 8:
        return dict(white=0.0, blue=0.0, brown=0.0, pink=0.0, purple=0.0)
    X = np.fft.rfft(x - np.mean(x))
    P = np.abs(X) ** 2
    freqs = np.fft.rfftfreq(n, d=1/fs)
    # avoid DC
    mask = freqs > 0
    freqs = freqs[mask]
    P = P[mask]
    if freqs.size < 5 or np.all(P == 0):
        return dict(white=0.0, blue=0.0, brown=0.0, pink=0.0, purple=0.0)

    logf = np.log(freqs)
    logP = np.log(P + 1e-12)

    def corr_with_alpha(alpha):
        # ideal: logP ~ -alpha * logf  (up to constant)
        target = -alpha * logf
        # correlation coefficient
        t = target - target.mean()
        s = logP - logP.mean()
        denom = np.linalg.norm(t) * np.linalg.norm(s)
        return float((t @ s) / denom) if denom > 0 else 0.0

    return {
        "white":  corr_with_alpha(0),
        "blue":   corr_with_alpha(-1),
        "brown":  corr_with_alpha(2),
        "pink":   corr_with_alpha(1),
        "purple": corr_with_alpha(-2),
    }

def extract_34_features(window: np.ndarray, fs: float = FS, wavelet: str = "db8"):
    """Return a 1D numpy array of length 34."""
    x = np.asarray(window, dtype=np.float64)

    # Group 1: basic stats (plus mean to make 34 total)
    f_min = float(np.min(x))
    f_max = float(np.max(x))
    f_mean = float(np.mean(x))
    f_var = float(np.var(x))
    f_skew = float(stats.skew(x)) if x.size > 2 else 0.0
    f_kurt = float(stats.kurtosis(x, fisher=False)) if x.size > 3 else 0.0
    f_iqr = float(np.percentile(x, 75) - np.percentile(x, 25))

    # Group 2
    hj_mob, hj_comp = _hjorth_params(x)
    ghe = _ghe_q1(x)
    went = _wavelet_packet_entropy(x, wavelet=wavelet, level=3)
    log_went = float(np.log(went + 1e-12))
    rms = float(np.sqrt(np.mean(x**2)))

    # Group 3 (shape factors)
    abs_mean = float(np.mean(np.abs(x))) + 1e-12
    peak = float(np.max(np.abs(x))) + 1e-12
    impulse = float(peak / abs_mean)
    margin = float(peak / ((float(np.mean(np.sqrt(np.abs(x)))) + 1e-12) ** 2))
    shape = float(rms / abs_mean)
    crest = float(peak / (rms + 1e-12))

    # Group 4 (frequency)
    f_cent, f_rms, f_rootvar = _spectral_features(x, fs)

    # Group 5 (color noise similarity)
    cn = _color_noise_similarity(x, fs)
    sim_white = cn["white"]
    sim_blue = cn["blue"]
    sim_brown = cn["brown"]
    sim_pink = cn["pink"]
    sim_purple = cn["purple"]

    # Group 6 (wavelet decomposition order 8, detail levels 1,4,8)
    coeffs = pywt.wavedec(x, wavelet=wavelet, level=8, mode="symmetric")
    # coeffs: [cA8, cD8, cD7, ..., cD1]
    cD1 = coeffs[-1]
    cD4 = coeffs[-4]
    cD8 = coeffs[1]

    wd1_min, wd1_max, wd1_mean = float(np.min(cD1)), float(np.max(cD1)), float(np.mean(cD1))
    wd4_min, wd4_max, wd4_mean = float(np.min(cD4)), float(np.max(cD4)), float(np.mean(cD4))
    wd8_min, wd8_max, wd8_mean = float(np.min(cD8)), float(np.max(cD8)), float(np.mean(cD8))

    feats = np.array([
        # Group 1 (7)
        f_min, f_max, f_mean, f_var, f_skew, f_kurt, f_iqr,
        # Group 2 (6)
        hj_mob, hj_comp, ghe, went, log_went, rms,
        # Group 3 (4)
        impulse, margin, shape, crest,
        # Group 4 (3)
        f_cent, f_rms, f_rootvar,
        # Group 5 (5)
        sim_white, sim_blue, sim_brown, sim_pink, sim_purple,
        # Group 6 (9)
        wd1_min, wd1_max, wd1_mean,
        wd4_min, wd4_max, wd4_mean,
        wd8_min, wd8_max, wd8_mean
    ], dtype=np.float64)

    assert feats.shape[0] == 34, f"Expected 34 features, got {feats.shape[0]}"
    return feats


## 3) Segment-based windowing

In [ ]:
def build_samples_from_signal(x: np.ndarray, fs: float, l_seconds: int, N: int):
    x = np.asarray(x, dtype=np.float64)
    l_samp = int(round(l_seconds * fs))
    seg_len = N * l_samp
    if x.size < seg_len:
        raise ValueError("Signal too short for chosen N and l.")

    end_indices = np.arange(seg_len - 1, x.size, l_samp)  # shift by l each step
    X_rows = []

    for end_idx in end_indices:
        seg = x[end_idx - seg_len + 1 : end_idx + 1]
        feats = []
        for k in range(N):
            w = seg[k*l_samp : (k+1)*l_samp]
            feats.append(extract_34_features(w, fs=fs))
        X_rows.append(np.concatenate(feats))  # 34*N

    X = np.vstack(X_rows)
    return X, end_indices


## 4) Labeling

In [ ]:
def make_labels_from_end_indices(end_indices: np.ndarray, fs: float = FS, normal_hours: int = 24) -> np.ndarray:
    boundary = int(normal_hours * 3600 * fs)
    end_indices = np.asarray(end_indices)
    return (end_indices >= boundary).astype(int)


## 5) enforce 274 samples per plant (for different window length l)(optional)

In [ ]:
def pick_274_balanced(X: np.ndarray, y: np.ndarray, n: int = 274, seed: int = 0):
    rng = np.random.default_rng(seed)
    idx0 = np.where(y == 0)[0]
    idx1 = np.where(y == 1)[0]
    k = n // 2
    if len(idx0) < k or len(idx1) < k:
        raise ValueError(f"Not enough samples to pick {k} per class (have {len(idx0)} and {len(idx1)}).")
    take0 = rng.choice(idx0, size=k, replace=False)
    take1 = rng.choice(idx1, size=k, replace=False)
    idx = np.concatenate([take0, take1])
    rng.shuffle(idx)
    return X[idx], y[idx]


## 6) Min–max normalization per plant

In [ ]:
def minmax_normalize_per_plant(X_plant: np.ndarray) -> np.ndarray:
    X_plant = np.asarray(X_plant, dtype=np.float64)
    min_vals = X_plant.min(axis=0)
    max_vals = X_plant.max(axis=0)
    denom = max_vals - min_vals
    denom[denom == 0] = 1.0
    return (X_plant - min_vals) / denom


## 7) Run preprocessing for all plants and concatenate

In [ ]:
import os, glob
import numpy as np

filtered_files = sorted(glob.glob(os.path.join(BASE_DIR, "*_filtered.npy")))
print("Found", len(filtered_files), "filtered files")

out_dir = os.path.join(BASE_DIR, f"preprocessed_l{L_SECONDS}_N{N_WINDOWS}")
os.makedirs(out_dir, exist_ok=True)

for i, path in enumerate(filtered_files):
    print(f"Processing plant {i+1}/{len(filtered_files)}: {os.path.basename(path)}")

    sig = np.load(path)  # already notch-filtered

    # 1) build samples
    Xp, end_idx = build_samples_from_signal(sig, fs=FS, l_seconds=L_SECONDS, N=N_WINDOWS)

    # 2) label by segment end time
    yp = make_labels_from_end_indices(end_idx, fs=FS, normal_hours=24)

    # 3) optional 274 selection
    if USE_274_PER_PLANT:
        Xp, yp = pick_274_balanced(Xp, yp, n=274, seed=SEED + i)

    # 4) normalize per plant
    Xp = minmax_normalize_per_plant(Xp)

    # 5) reduce memory and disk size
    Xp = Xp.astype(np.float32)
    yp = yp.astype(np.int8)

    # 6) save PER PLANT
    np.save(os.path.join(out_dir, f"X_plant_{i:02d}.npy"), Xp)
    np.save(os.path.join(out_dir, f"y_plant_{i:02d}.npy"), yp)

    # free memory
    del sig, Xp, yp

print("Saved per-plant preprocessed arrays to:", out_dir)


Found 36 filtered files
Processing plant 1/36: generalstress_deficitCa2+_channelF4_filtered.npy
Processing plant 2/36: generalstress_deficitCa2+_channelF5_filtered.npy
Processing plant 3/36: generalstress_deficitCa2+_channelG2_filtered.npy
Processing plant 4/36: generalstress_deficitFe2+_channelF4_filtered.npy
Processing plant 5/36: generalstress_deficitFe2+_channelF7_filtered.npy
Processing plant 6/36: generalstress_deficitFe2+_channelG3_filtered.npy
Processing plant 7/36: generalstress_deficitMn2+_channelB4_filtered.npy
Processing plant 8/36: generalstress_deficitMn2+_channelB7_filtered.npy
Processing plant 9/36: generalstress_deficitMn2+_channelC2_filtered.npy
Processing plant 10/36: generalstress_deficitN_channelB1_filtered.npy
Processing plant 11/36: generalstress_deficitN_channelB6_filtered.npy
Processing plant 12/36: generalstress_deficitN_channelC2_filtered.npy
Processing plant 13/36: generalstress_drought_channelG1B0_filtered.npy
Processing plant 14/36: generalstress_drought_c

In [ ]:
import os, glob
import numpy as np

# load all saved per-plant arrays
X_files = sorted(glob.glob(os.path.join(out_dir, "X_plant_*.npy")))
y_files = sorted(glob.glob(os.path.join(out_dir, "y_plant_*.npy")))

assert len(X_files) == len(y_files), "Mismatch between X and y files count"

X_list, y_list = [], []

for xf, yf in zip(X_files, y_files):
    X_list.append(np.load(xf))
    y_list.append(np.load(yf))

X_all = np.concatenate(X_list, axis=0)
y_all = np.concatenate(y_list, axis=0)

np.save(os.path.join(out_dir, "X_all.npy"), X_all)
np.save(os.path.join(out_dir, "y_all.npy"), y_all)

print("Saved combined dataset:")
print("X_all:", X_all.shape)
print("y_all:", y_all.shape)


Saved combined dataset:
X_all: (103148, 510)
y_all: (103148,)


In [ ]:
import os

print("out_dir =", os.path.abspath(out_dir))


out_dir = /content/GeneralStressPaperData/preprocessed_l60_N15


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!mkdir -p /content/drive/MyDrive/GeneralStressPaperData
!cp -r /content/GeneralStressPaperData/preprocessed_l60_N15 \
      /content/drive/MyDrive/GeneralStressPaperData/


In [ ]:
import os

drive_dir = "/content/drive/MyDrive/GeneralStressPaperData/preprocessed_l60_N15"
print(os.listdir(drive_dir))


['X_plant_00.npy', 'y_plant_00.npy', 'X_plant_01.npy', 'y_plant_01.npy', 'X_plant_02.npy', 'y_plant_02.npy', 'X_plant_03.npy', 'y_plant_03.npy', 'X_plant_04.npy', 'y_plant_04.npy', 'X_plant_05.npy', 'y_plant_05.npy', 'X_plant_06.npy', 'y_plant_06.npy', 'X_plant_07.npy', 'y_plant_07.npy', 'X_plant_08.npy', 'y_plant_08.npy', 'X_plant_09.npy', 'y_plant_09.npy', 'X_plant_10.npy', 'y_plant_10.npy', 'X_plant_11.npy', 'y_plant_11.npy', 'X_plant_12.npy', 'y_plant_12.npy', 'X_plant_13.npy', 'y_plant_13.npy', 'X_plant_14.npy', 'y_plant_14.npy', 'X_plant_15.npy', 'y_plant_15.npy', 'X_plant_16.npy', 'y_plant_16.npy', 'X_plant_17.npy', 'y_plant_17.npy', 'X_plant_18.npy', 'y_plant_18.npy', 'X_plant_19.npy', 'y_plant_19.npy', 'X_plant_20.npy', 'y_plant_20.npy', 'X_plant_21.npy', 'y_plant_21.npy', 'X_plant_22.npy', 'y_plant_22.npy', 'X_plant_23.npy', 'y_plant_23.npy', 'X_plant_24.npy', 'y_plant_24.npy', 'X_plant_25.npy', 'y_plant_25.npy', 'X_plant_26.npy', 'y_plant_26.npy', 'X_plant_27.npy', 'y_plant_